# Análise Preditiva de Demanda
Projeto em Business Intelligence e Analytics — Distribuidora Farmacêutica

Prevê a demanda futura de cada produto a partir do histórico de vendas (`data/processed/fato_vendas.csv`), compara o modelo contra uma baseline simples (média móvel) e gera `data/processed/previsao_demanda.csv`, usado na página de Previsão de Demanda do dashboard.

**Pré-requisito:** rode antes o notebook `analise_exploratoria.ipynb` (ou o script `scripts/tratamento_dados.py`) para gerar os arquivos em `data/processed/`.

In [ ]:
!pip install -q scikit-learn matplotlib pandas numpy


## Carga e diagnóstico inicial dos dados
Carrega as tabelas `fato_vendas` e `fato_estoque` já processadas pelo pipeline de ETL e faz uma checagem inicial de período coberto, nulos, duplicidades e devoluções.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

ARQ_VENDAS = "data/processed/fato_vendas.csv"
ARQ_ESTOQUE = "data/processed/fato_estoque.csv"

vendas = pd.read_csv(ARQ_VENDAS)
estoque = pd.read_csv(ARQ_ESTOQUE)

print("Vendas:", vendas.shape)
print("Estoque:", estoque.shape)

display(vendas.head())
display(estoque.head())

print("VENDAS")
print(vendas.info())

print("ESTOQUE")
print(estoque.info())

print("PERÍODO DE VENDAS")

vendas["competencia"] = pd.to_datetime(vendas["competencia"])

print("Primeira competência:",
      vendas["competencia"].min())

print("Última competência:",
      vendas["competencia"].max())

print("\nQuantidade de meses:",
      vendas["competencia"].nunique())

print("\nQuantidade de EANs:",
      vendas["ean"].nunique())

# Qualidade dos dados

print("VALORES NULOS")

print(
    vendas.isnull()
    .sum()
    .sort_values(ascending=False)
)

print("DUPLICIDADES")

print(
    "Linhas duplicadas:",
    vendas.duplicated().sum()
)

print("UNIDADES")

print(vendas["unidades"].describe())

print("\nQuantidade de vendas negativas:",
      (vendas["unidades"] < 0).sum())

print("DEVOLUÇÕES")

print(
    vendas["flag_devolucao"]
    .value_counts(dropna=False)
)

print("OUTLIERS")

print(
    vendas["flag_outlier_valor"]
    .value_counts(dropna=False)
)

vendas_modelo = vendas.copy()

# Garantir tipos
vendas_modelo["competencia"] = pd.to_datetime(
    vendas_modelo["competencia"]
)

vendas_modelo["unidades"] = pd.to_numeric(
    vendas_modelo["unidades"],
    errors="coerce"
)

# Tirar registros sem EAN ou competência
vendas_modelo = vendas_modelo.dropna(
    subset=["ean", "competencia"]
)

# Tirar devoluções
vendas_modelo = vendas_modelo[
    vendas_modelo["flag_devolucao"] != True
].copy()

print("Registros após tratamento:",
      len(vendas_modelo))


## Série Temporal Mensal
Agrupa as vendas por produto e mês, consolidando a série que será usada para treinar o modelo.


In [ ]:
vendas_mensais = (
    vendas_modelo
    .groupby(
        ["ean", "competencia"],
        as_index=False
    )
    .agg(
        unidades=("unidades", "sum"),
        valor=("valor", "sum")
    )
)

vendas_mensais = vendas_mensais.sort_values(
    ["ean", "competencia"]
)

print("Formato:",
      vendas_mensais.shape)

display(vendas_mensais.head(20))


## Cobertura Temporal Dos Produtos
Verifica quantos meses de histórico cada produto tem — essencial para saber se há dado suficiente para prever.


In [ ]:
historico_produtos = (
    vendas_mensais
    .groupby("ean")
    .agg(
        meses_com_venda=("competencia", "nunique"),
        primeira_venda=("competencia", "min"),
        ultima_venda=("competencia", "max"),
        venda_total=("unidades", "sum")
    )
    .reset_index()
)

display(
    historico_produtos[
        ["meses_com_venda", "venda_total"]
    ].describe()
)


## Completar Meses Sem Venda
Preenche os meses sem venda com zero, para que a série temporal de cada produto fique contínua (sem 'buracos').


In [ ]:
data_inicio = vendas_mensais["competencia"].min()
data_fim = vendas_mensais["competencia"].max()

meses = pd.date_range(
    start=data_inicio,
    end=data_fim,
    freq="MS"
)

eans = vendas_mensais["ean"].unique()

calendario_produtos = pd.MultiIndex.from_product(
    [eans, meses],
    names=["ean", "competencia"]
).to_frame(index=False)

serie_completa = calendario_produtos.merge(
    vendas_mensais,
    on=["ean", "competencia"],
    how="left"
)

serie_completa["unidades"] = (
    serie_completa["unidades"]
    .fillna(0)
)

serie_completa["valor"] = (
    serie_completa["valor"]
    .fillna(0)
)

serie_completa = serie_completa.sort_values(
    ["ean", "competencia"]
)

print("Base completa:",
      serie_completa.shape)

display(serie_completa.head(20))


## Features Temporais
Cria as variáveis de entrada do modelo: vendas defasadas (lag 1, 2, 3 e 6 meses), médias móveis de 3 e 6 meses, e o mês/ano da competência.


In [ ]:
serie_modelo = serie_completa.copy()

grupo = serie_modelo.groupby("ean")["unidades"]

serie_modelo["lag_1"] = grupo.shift(1)
serie_modelo["lag_2"] = grupo.shift(2)
serie_modelo["lag_3"] = grupo.shift(3)
serie_modelo["lag_6"] = grupo.shift(6)

serie_modelo["media_3m"] = (
    grupo
    .shift(1)
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

serie_modelo["media_6m"] = (
    grupo
    .shift(1)
    .rolling(6)
    .mean()
    .reset_index(level=0, drop=True)
)

# Variáveis temporais
serie_modelo["mes"] = (
    serie_modelo["competencia"].dt.month
)

serie_modelo["ano"] = (
    serie_modelo["competencia"].dt.year
)

display(serie_modelo.head(15))


## Baseline
Calcula um modelo de referência simples (média móvel de 3 meses) para servir de comparação — todo modelo de Machine Learning deve superar uma baseline ingênua para se justificar.


In [ ]:
baseline = serie_modelo.dropna(
    subset=["media_3m"]
).copy()

mae_baseline = mean_absolute_error(
    baseline["unidades"],
    baseline["media_3m"]
)

rmse_baseline = np.sqrt(
    mean_squared_error(
        baseline["unidades"],
        baseline["media_3m"]
    )
)

print("BASELINE - MÉDIA MÓVEL 3 MESES")
print("--------------------------------")
print("MAE :", round(mae_baseline, 2))
print("RMSE:", round(rmse_baseline, 2))


## Treino E Teste
Divide a série em treino (histórico mais antigo) e teste (últimos 6 meses) — a divisão respeita a ordem cronológica, sem embaralhar os dados.


In [ ]:
# Remover linhas sem histórico suficiente
dados_ml = serie_modelo.dropna(
    subset=[
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_6",
        "media_3m",
        "media_6m"
    ]
).copy()

# Últimos 6 meses como teste
data_corte = (
    dados_ml["competencia"].max()
    - pd.DateOffset(months=5)
)

treino = dados_ml[
    dados_ml["competencia"] < data_corte
].copy()

teste = dados_ml[
    dados_ml["competencia"] >= data_corte
].copy()

print("Treino:", treino.shape)
print("Teste :", teste.shape)

print("\nTreino:",
      treino["competencia"].min(),
      "até",
      treino["competencia"].max())

print("Teste:",
      teste["competencia"].min(),
      "até",
      teste["competencia"].max())


## Random Forest
Treina um Random Forest Regressor com as features temporais para prever a quantidade de unidades vendidas.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

features = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "media_3m",
    "media_6m",
    "mes"
]

target = "unidades"

X_train = treino[features]
y_train = treino[target]

X_test = teste[features]
y_test = teste[target]

modelo_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(
    X_train,
    y_train
)

previsao_rf = modelo_rf.predict(
    X_test
)

previsao_rf = np.maximum(
    previsao_rf,
    0
)


## Avaliação
Compara o erro do Random Forest com o da baseline usando MAE e RMSE.


In [ ]:
mae_rf = mean_absolute_error(
    y_test,
    previsao_rf
)

rmse_rf = np.sqrt(
    mean_squared_error(
        y_test,
        previsao_rf
    )
)

print("RANDOM FOREST")
print("--------------------------------")
print("MAE :", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))

print("\nBASELINE")
print("--------------------------------")
print("MAE :", round(mae_baseline, 2))
print("RMSE:", round(rmse_baseline, 2))


## Comparação
Tabela resumo comparando os dois modelos lado a lado.


In [ ]:
comparacao = pd.DataFrame({
    "Modelo": [
        "Média móvel 3 meses",
        "Random Forest"
    ],
    "MAE": [
        mae_baseline,
        mae_rf
    ],
    "RMSE": [
        rmse_baseline,
        rmse_rf
    ]
})

display(
    comparacao.sort_values("MAE")
)


## Avaliação Detalhada Do Modelo
Calcula o erro (absoluto e percentual) previsão a previsão, para inspecionar caso a caso.


In [ ]:
resultado_teste = teste[
    ["ean", "competencia", "unidades", "media_3m"]
].copy()

resultado_teste["previsao_rf"] = previsao_rf

# Erros
resultado_teste["erro"] = (
    resultado_teste["unidades"]
    - resultado_teste["previsao_rf"]
)

resultado_teste["erro_abs"] = (
    resultado_teste["erro"].abs()
)

# Erro percentual
resultado_teste["erro_percentual"] = np.where(
    resultado_teste["unidades"] != 0,
    resultado_teste["erro_abs"]
    / resultado_teste["unidades"] * 100,
    np.nan
)

display(
    resultado_teste.describe()
)


## Produtos Com Maior Erro
Rankeia os produtos onde o modelo mais erra — útil para identificar categorias que precisam de mais atenção ou de um modelo diferente.


In [ ]:
erro_por_ean = (
    resultado_teste
    .groupby("ean")
    .agg(
        venda_real_media=("unidades", "mean"),
        previsao_media=("previsao_rf", "mean"),
        erro_medio=("erro", "mean"),
        erro_absoluto_medio=("erro_abs", "mean"),
        erro_percentual_medio=("erro_percentual", "mean")
    )
    .reset_index()
)

erro_por_ean = erro_por_ean.sort_values(
    "erro_absoluto_medio",
    ascending=False
)

display(
    erro_por_ean.head(20)
)

# Random Forest x Média Móvel

mae_baseline_teste = mean_absolute_error(
    teste["unidades"],
    teste["media_3m"]
)

rmse_baseline_teste = np.sqrt(
    mean_squared_error(
        teste["unidades"],
        teste["media_3m"]
    )
)

comparacao_modelos = pd.DataFrame({
    "Modelo": [
        "Média Móvel 3 meses",
        "Random Forest"
    ],
    "MAE": [
        mae_baseline_teste,
        mean_absolute_error(teste["unidades"], previsao_rf)
    ],
    "RMSE": [
        rmse_baseline_teste,
        np.sqrt(mean_squared_error(teste["unidades"], previsao_rf))
    ]
})

comparacao_modelos["Melhoria_RMSE_%"] = (
    1 -
    comparacao_modelos["RMSE"] /
    comparacao_modelos.loc[0, "RMSE"]
) * 100

display(comparacao_modelos)


## Wape
Calcula WAPE e sMAPE, duas métricas de erro percentual mais robustas que o MAE/RMSE para comparar produtos com volumes muito diferentes entre si.


In [ ]:
wape_rf = (
    np.abs(teste["unidades"] - previsao_rf).sum()
    / teste["unidades"].sum()
) * 100

wape_ma = (
    np.abs(teste["unidades"] - teste["media_3m"]).sum()
    / teste["unidades"].sum()
) * 100

print(f"WAPE - Random Forest: {wape_rf:.2f}%")
print(f"WAPE - Média Móvel: {wape_ma:.2f}%")

# sMAPE

y_real = teste["unidades"].values
y_rf = previsao_rf

smape_rf = (
    np.mean(
        2 * np.abs(y_real - y_rf) /
        (np.abs(y_real) + np.abs(y_rf) + 1e-8)
    )
) * 100

y_ma = teste["media_3m"].values

smape_ma = (
    np.mean(
        2 * np.abs(y_real - y_ma) /
        (np.abs(y_real) + np.abs(y_ma) + 1e-8)
    )
) * 100

print(f"sMAPE - Random Forest: {smape_rf:.2f}%")
print(f"sMAPE - Média Móvel: {smape_ma:.2f}%")


## Previsão X Realizado
Plota, para um produto de exemplo, a série realizada versus a prevista pelo modelo no período de teste.


In [ ]:
resultado_teste = teste[
    ["ean", "competencia", "unidades"]
].copy()

resultado_teste["previsao_rf"] = previsao_rf

# Escolher um SKU para visualizar
ean_exemplo = resultado_teste["ean"].iloc[0]

grafico = resultado_teste[
    resultado_teste["ean"] == ean_exemplo
].copy()

plt.figure(figsize=(12, 5))

plt.plot(
    grafico["competencia"],
    grafico["unidades"],
    marker="o",
    label="Realizado"
)

plt.plot(
    grafico["competencia"],
    grafico["previsao_rf"],
    marker="o",
    label="Previsão"
)

plt.title(
    f"Demanda real x prevista - EAN {ean_exemplo}"
)

plt.xlabel("Competência")
plt.ylabel("Unidades")

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


## Treinamento Final
Retreina o modelo com todo o histórico disponível (treino + teste), para gerar a versão final que fará as previsões futuras.


In [ ]:
modelo_final = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

modelo_final.fit(
    dados_ml[features],
    dados_ml[target]
)

print("Modelo final treinado.")


## Previsão Dos Próximos 3 Meses
Gera a previsão mês a mês para os 3 meses seguintes ao último disponível, alimentando o próprio modelo com suas previsões anteriores (previsão recursiva), e exporta o resultado em `data/processed/previsao_demanda.csv`.


In [ ]:
import pandas as pd
import numpy as np

# Quantidade de meses
n_meses = 3

# Último mês disponível no histórico
ultima_data = serie_modelo["competencia"].max()

# Meses futuros
datas_futuras = pd.date_range(
    start=ultima_data + pd.DateOffset(months=1),
    periods=n_meses,
    freq="MS"
)

previsoes_futuras = []

serie_previsao = serie_modelo[
    ["ean", "competencia", "unidades"]
].copy()

serie_previsao = serie_previsao.sort_values(
    ["ean", "competencia"]
).reset_index(drop=True)

# Prever mês a mês
for data_futura in datas_futuras:

    linhas_previsao = []

    for ean, grupo in serie_previsao.groupby("ean"):

        grupo = grupo.sort_values("competencia")

        ultimos_valores = grupo["unidades"].tail(6).tolist()

        # Garantir histórico suficiente
        if len(ultimos_valores) < 6:
            continue

        lag_1 = ultimos_valores[-1]
        lag_2 = ultimos_valores[-2]
        lag_3 = ultimos_valores[-3]
        lag_6 = ultimos_valores[-6]

        media_3m = np.mean(ultimos_valores[-3:])
        media_6m = np.mean(ultimos_valores[-6:])

        mes = data_futura.month

        X_futuro = pd.DataFrame({
            "lag_1": [lag_1],
            "lag_2": [lag_2],
            "lag_3": [lag_3],
            "lag_6": [lag_6],
            "media_3m": [media_3m],
            "media_6m": [media_6m],
            "mes": [mes]
        })

        previsao = modelo_rf.predict(X_futuro)[0]

        # Não permitir previsão negativa
        previsao = max(previsao, 0)

        linhas_previsao.append({
            "ean": ean,
            "competencia": data_futura,
            "previsao_demanda": previsao
        })

    previsao_mes = pd.DataFrame(linhas_previsao)

    previsoes_futuras.append(previsao_mes)

    novas_linhas = previsao_mes.rename(
        columns={"previsao_demanda": "unidades"}
    )[["ean", "competencia", "unidades"]]

    serie_previsao = pd.concat(
        [serie_previsao, novas_linhas],
        ignore_index=True
    )

# Consolidar todas as previsões
previsao_demanda = pd.concat(
    previsoes_futuras,
    ignore_index=True
)

previsao_demanda = previsao_demanda.sort_values(
    ["ean", "competencia"]
).reset_index(drop=True)

print("Previsões geradas:")
display(previsao_demanda.head(20))

print("\nQuantidade de registros:")
print(len(previsao_demanda))

print("\nPeríodos previstos:")
print(previsao_demanda["competencia"].unique())

previsao_demanda["previsao_demanda"] = (
    previsao_demanda["previsao_demanda"]
    .round(0)
    .astype(int)
)

# Salvar CSV
previsao_demanda.to_csv(
    "data/processed/previsao_demanda.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo com sucesso.")
